# Assignment 2 Experiment Notebook

## Project
A Trustworthiness Audit of Open-Source LLMs for Phishing-Support Advice

## Purpose of this notebook
This notebook loads the benchmark prompt files, checks their structure, combines them into one master benchmark file, and prepares the experiment pipeline for evaluating 3 open-source LLMs.

## Models
- Llama 3.1 8B
- Qwen2.5 7B
- Mistral 7B Instruct

## Benchmark
- Task A: Detection = 100 prompts
- Task B: Recovery = 60 prompts
- Task C: Fairness = 80 prompts
- Task D: Robustness = 60 prompts
- Total prompts = 300
- Total outputs expected = 900

In [2]:
!pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.7 MB 14.2 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.7 MB 14.0 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   -------- ------------------------------- 2.6/12.3 MB 14.8 MB/s eta 0:00:01
   -------------- ------------------------- 4.5/12.3 MB 10.8 MB/s eta 0:00:01
   ---------------------- ----------------- 6.8/12.3 MB 11.2 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.3 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------  12.1/12.3 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 10.7 MB/s  0:00:01

   --------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

In [8]:

PROJECT_ROOT = Path.cwd().resolve().parent

PROMPTS_DIR = PROJECT_ROOT / "prompts"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SCORING_DIR = DATA_DIR / "scoring"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
EXPORTS_DIR = PROJECT_ROOT / "notebook" / "exports"

for folder in [RAW_DIR, PROCESSED_DIR, SCORING_DIR, OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR, EXPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Prompts folder:", PROMPTS_DIR)
print("Processed data folder:", PROCESSED_DIR)
print("Notebook exports folder:", EXPORTS_DIR)

Project root: C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu
Prompts folder: C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\prompts
Processed data folder: C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\data\processed
Notebook exports folder: C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\notebook\exports


In [9]:
taskA_file = PROMPTS_DIR / "taskA_detection.csv"
taskB_file = PROMPTS_DIR / "taskB_recovery.csv"
taskC_file = PROMPTS_DIR / "taskC_fairness.csv"
taskD_file = PROMPTS_DIR / "taskD_robustness.csv"

print(taskA_file)
print(taskB_file)
print(taskC_file)
print(taskD_file)

C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\prompts\taskA_detection.csv
C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\prompts\taskB_recovery.csv
C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\prompts\taskC_fairness.csv
C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\prompts\taskD_robustness.csv


In [10]:
df_a = pd.read_csv(taskA_file)
df_b = pd.read_csv(taskB_file)
df_c = pd.read_csv(taskC_file)
df_d = pd.read_csv(taskD_file)

print("Task A shape:", df_a.shape)
print("Task B shape:", df_b.shape)
print("Task C shape:", df_c.shape)
print("Task D shape:", df_d.shape)

Task A shape: (100, 13)
Task B shape: (60, 11)
Task C shape: (80, 13)
Task D shape: (60, 12)


In [11]:
print("TASK A PREVIEW")
display(df_a.head(3))

print("TASK B PREVIEW")
display(df_b.head(3))

print("TASK C PREVIEW")
display(df_c.head(3))

print("TASK D PREVIEW")
display(df_d.head(3))

TASK A PREVIEW


,prompt_id,task,scenario_group,comparison_group,polarity,message_channel,sender_context,message_text,ground_truth_label,primary_signal,secondary_signal,difficulty,notes
0,A001,detection,bank_alert_phish,det_bank_alert_phish,phishing,email,Claims to be from a bank,Your account has been limited due to unusual a...,phishing,urgency,credential_request,medium,bank phishing 1
1,A002,detection,bank_alert_phish,det_bank_alert_phish,phishing,email,Claims to be from a bank security team,We detected suspicious transactions on your ac...,phishing,threat_of_loss,credential_request,medium,bank phishing 2
2,A003,detection,bank_alert_phish,det_bank_alert_phish,phishing,sms,Claims to be from a bank,Bank alert: your online access will be suspend...,phishing,urgency,account_verification_link,medium,bank phishing 3


TASK B PREVIEW


,prompt_id,task,incident_type,comparison_group,user_persona,user_context,scenario_text,expected_risk_label,gold_actions,difficulty,notes
0,B001,recovery,clicked_link,rec_clicked_link,general_user,Received a fake bank email,I clicked the link in a banking email but clos...,phishing_related,stop_engagement;verify_official_channel;scan_d...,medium,clicked link 1
1,B002,recovery,clicked_link,rec_clicked_link,general_user,Opened a parcel delivery link,I tapped a delivery text message link because ...,phishing_related,stop_engagement;verify_official_channel;scan_d...,medium,clicked link 2
2,B003,recovery,clicked_link,rec_clicked_link,general_user,Opened a university account link,I clicked a student email link saying my unive...,phishing_related,stop_engagement;verify_official_channel;scan_d...,medium,clicked link 3


TASK C PREVIEW


,prompt_id,task,pair_id,persona_dimension,comparison_group,base_scenario_id,persona_variant,persona_details,scenario_text,expected_risk_label,gold_actions,difficulty,notes
0,C001,fairness,F001,age,fair_age,BASE_AGE_001,older_adult,age=72|digital_confidence=medium|anxiety=medium,I am 72 years old. I entered my password into ...,phishing_related,change_password;contact_bank;enable_mfa;check_...,high,age pair 1 older
1,C002,fairness,F001,age,fair_age,BASE_AGE_001,young_adult,age=22|digital_confidence=medium|anxiety=medium,I am 22 years old. I entered my password into ...,phishing_related,change_password;contact_bank;enable_mfa;check_...,high,age pair 1 younger
2,C003,fairness,F002,age,fair_age,BASE_AGE_002,older_adult,age=70|digital_confidence=medium|anxiety=medium,I am 70 years old. I shared a one-time code fr...,phishing_related,contact_bank;secure_account;check_transactions...,high,age pair 2 older


TASK D PREVIEW


,prompt_id,task,pair_id,manipulation_family,comparison_group,base_scenario_id,variant_type,scenario_text,ground_truth_label,gold_actions,difficulty,notes
0,D001,robustness,R001,urgency_typos,rob_urgency_typos,BASE_URG_001,clean,I got an email from my bank asking me to confi...,phishing,warn;do_not_click;verify_official_channel;repo...,easy,urgency pair 1 clean
1,D002,robustness,R001,urgency_typos,rob_urgency_typos,BASE_URG_001,manipulated,plz help my bank says ACT NOW or acct gets loc...,phishing,warn;do_not_click;verify_official_channel;repo...,hard,urgency pair 1 manipulated
2,D003,robustness,R002,urgency_typos,rob_urgency_typos,BASE_URG_002,clean,I received a text saying my parcel cannot be d...,phishing,warn;do_not_click;verify_official_channel;repo...,easy,urgency pair 2 clean


In [12]:
task_counts = pd.DataFrame({
    "task_file": ["taskA_detection", "taskB_recovery", "taskC_fairness", "taskD_robustness"],
    "num_prompts": [len(df_a), len(df_b), len(df_c), len(df_d)]
})

task_counts["expected_outputs_for_3_models"] = task_counts["num_prompts"] * 3

display(task_counts)

print("Total prompts:", task_counts["num_prompts"].sum())
print("Total expected outputs across 3 models:", task_counts["expected_outputs_for_3_models"].sum())

,task_file,num_prompts,expected_outputs_for_3_models
0,taskA_detection,100,300
1,taskB_recovery,60,180
2,taskC_fairness,80,240
3,taskD_robustness,60,180


Total prompts: 300
Total expected outputs across 3 models: 900


In [13]:
print("Task A groups")
display(df_a["scenario_group"].value_counts().sort_index())

print("Task B incident types")
display(df_b["incident_type"].value_counts().sort_index())

print("Task C persona dimensions")
display(df_c["persona_dimension"].value_counts().sort_index())

print("Task D manipulation families")
display(df_d["manipulation_family"].value_counts().sort_index())

Task A groups


scenario_group
bank_alert_phish             10
bank_notice_legit            10
delivery_notice_legit        10
delivery_phish               10
government_tax_phish         10
job_offer_phish              10
subscription_notice_legit    10
university_login_phish       10
university_notice_legit      10
workplace_notice_legit       10
Name: count, dtype: int64

Task B incident types


incident_type
attachment_opened        10
clicked_link             10
otp_shared               10
password_entered         10
personal_info_replied    10
unsure_before_acting     10
Name: count, dtype: int64

Task C persona dimensions


persona_dimension
age                      20
anxiety_level            20
digital_confidence       20
prior_scam_experience    20
Name: count, dtype: int64

Task D manipulation families


manipulation_family
authority_pressure      20
multi_turn_confusion    20
urgency_typos           20
Name: count, dtype: int64

In [14]:
df_a["prompt_text"] = df_a["message_text"]
df_b["prompt_text"] = df_b["scenario_text"]
df_c["prompt_text"] = df_c["scenario_text"]
df_d["prompt_text"] = df_d["scenario_text"]

print("Common prompt_text column added.")

Common prompt_text column added.


In [15]:
common_columns = ["prompt_id", "task", "comparison_group", "prompt_text", "difficulty", "notes"]

df_a_common = df_a[common_columns].copy()
df_b_common = df_b[common_columns].copy()
df_c_common = df_c[common_columns].copy()
df_d_common = df_d[common_columns].copy()

master_prompts = pd.concat(
    [df_a_common, df_b_common, df_c_common, df_d_common],
    ignore_index=True
)

display(master_prompts.head())
print("Master prompt file shape:", master_prompts.shape)

,prompt_id,task,comparison_group,prompt_text,difficulty,notes
0,A001,detection,det_bank_alert_phish,Your account has been limited due to unusual a...,medium,bank phishing 1
1,A002,detection,det_bank_alert_phish,We detected suspicious transactions on your ac...,medium,bank phishing 2
2,A003,detection,det_bank_alert_phish,Bank alert: your online access will be suspend...,medium,bank phishing 3
3,A004,detection,det_bank_alert_phish,A new device attempted to access your account....,medium,bank phishing 4
4,A005,detection,det_bank_alert_phish,Your account requires immediate identity confi...,hard,bank phishing 5


Master prompt file shape: (300, 6)


In [16]:
master_prompt_file = PROCESSED_DIR / "master_benchmark_prompts.csv"
master_prompts.to_csv(master_prompt_file, index=False)

print("Saved master prompt file to:")
print(master_prompt_file)

Saved master prompt file to:
C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\data\processed\master_benchmark_prompts.csv


In [17]:
benchmark_summary = {
    "project_title": "A Trustworthiness Audit of Open-Source LLMs for Phishing-Support Advice",
    "models": ["llama3.1:8b", "qwen2.5:7b", "mistral:7b-instruct"],
    "task_counts": {
        "taskA_detection": int(len(df_a)),
        "taskB_recovery": int(len(df_b)),
        "taskC_fairness": int(len(df_c)),
        "taskD_robustness": int(len(df_d))
    },
    "total_prompts": int(len(df_a) + len(df_b) + len(df_c) + len(df_d)),
    "expected_outputs": int((len(df_a) + len(df_b) + len(df_c) + len(df_d)) * 3),
    "generated_at": datetime.now().isoformat()
}

summary_file = EXPORTS_DIR / "benchmark_summary.json"

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(benchmark_summary, f, indent=4)

print("Saved benchmark summary to:")
print(summary_file)

Saved benchmark summary to:
C:\Users\Admin\OneDrive\Documents\ARTI6000\assignment2_a1943902_Sahu\notebook\exports\benchmark_summary.json


In [18]:
print("Notebook setup complete.")
print("Total prompts loaded:", len(master_prompts))
print("Expected outputs across 3 models:", len(master_prompts) * 3)
print("Master CSV exists:", master_prompt_file.exists())
print("Summary JSON exists:", summary_file.exists())

Notebook setup complete.
Total prompts loaded: 300
Expected outputs across 3 models: 900
Master CSV exists: True
Summary JSON exists: True
